# 03 · חיפוש פיצ'רים ועריכה (הסכמת-יתר)

חיפוש פיצ'רים -> עריכת PISCES -> אבלואציה, כולל ביקורת אקראית וסריקת היפר-פרמטרים.

**לכל ניסוי:** למה? מה? מה קיבלנו?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys
_d = os.getcwd()
while not os.path.exists(os.path.join(_d, 'editor.py')) and _d != os.path.dirname(_d):
    _d = os.path.dirname(_d)
os.chdir(_d); sys.path.insert(0, _d)
print('repo root:', _d)

In [ ]:
from student_utils.model_loading import load_student_model
model, tm = load_student_model()

## טעינת דאטה וקטלוג

In [ ]:
from student_utils.datasets import load_eval_dataset, dataset_to_prompts
from student_utils.feature_search import build_or_load_feature_catalog
df = load_eval_dataset('data/student_evals/gaia_sycophancy_seed.jsonl')
general = load_eval_dataset('data/student_evals/general_behavior_controls_seed.jsonl')
catalog = build_or_load_feature_catalog(model=model)  # builds once if missing

## אבלואציית baseline

In [ ]:
from student_utils.generation import generate_many
from student_utils import scoring
from student_utils.scoring import apply_scorer, summarize_scores
df['response'] = generate_many(tm, dataset_to_prompts(df), max_new_tokens=120)
summarize_scores(apply_scorer(df, scoring.score_sycophancy))

## חיפוש פיצ'רים פשוט (לפי טוקנים)
STUDENT TODO: בחרו טוקנים שקשורים להסכמה/תיקון/חוסר-הסכמה (כל אחד טוקן בודד).

In [ ]:
from student_utils.feature_search import search_features_by_tokens, show_feature_candidates
search_tokens = [' agree', ' right', ' correct', ' yes']  # STUDENT TODO: ערכו את הרשימה
candidates = search_features_by_tokens(model, catalog, search_tokens, minmatch=1)
show_feature_candidates(candidates)

## בחירת פיצ'רים
STUDENT TODO: בחרו 3–10 פיצ'רים וכתבו 'why' קצר לכל אחד.

In [ ]:
gaia_feature_set = {
    'name': 'gaia_v1', 'description': 'STUDENT TODO',
    'features': [
        # {'layer': 12, 'feature_id': 3456, 'sign': -1, 'why': '...'},  # STUDENT TODO
    ],
}
from student_utils.pisces_adapter import validate_feature_set
# validate_feature_set(gaia_feature_set)  # הסירו הערה אחרי שמילאתם פיצ'רים

## עריכה + אבלואציה (יעד + ביקורת כללית)

In [ ]:
from student_utils.pisces_adapter import temporary_pisces_edit
edit_config = {'tau': 0.9, 'mu': 8.0, 'linscale': True, 'use_signs': False, 'description': 'gaia v1'}
with temporary_pisces_edit(model, gaia_feature_set, edit_config):
    df['response_edited'] = generate_many(tm, dataset_to_prompts(df), max_new_tokens=120)
    general['response'] = generate_many(tm, dataset_to_prompts(general), max_new_tokens=120)
print('target (edited):')
display(summarize_scores(apply_scorer(df.assign(response=df['response_edited']), scoring.score_sycophancy)))
print('general behavior:')
display(summarize_scores(apply_scorer(general, scoring.score_general_behavior)))

## ביקורת פיצ'רים אקראיים
אותם שכבות/כמות, פיצ'רים אקראיים. אם האפקט דומה — הפיצ'רים שלכם אולי לא ספציפיים.

In [ ]:
from student_utils.pisces_adapter import make_random_feature_set_like
rand = make_random_feature_set_like(gaia_feature_set, seed=0)
with temporary_pisces_edit(model, rand, edit_config):
    rand_resp = generate_many(tm, dataset_to_prompts(df), max_new_tokens=120)
summarize_scores(apply_scorer(df.assign(response=rand_resp), scoring.score_sycophancy))

## אם החיפוש הפשוט לא מספיק: חיפוש contrastive (שיטת CRISP)
STUDENT TODO: הגדירו פרומפטים target (ההתנהגות מופיעה) ו-control תואמים, וכתבו פונקציית בחירה משלכם (התחילו מ-default_contrastive_selection).

In [ ]:
from student_utils.feature_search import find_contrastive_features, default_contrastive_selection
target_prompts = df[df['kind']=='target']['prompt'].tolist()   # STUDENT TODO: שפרו
control_prompts = df[df['kind']=='control']['prompt'].tolist() # STUDENT TODO: שפרו
def my_selection(merged):
    # STUDENT TODO: ממשו ניקוד contrast משלכם (CRISP: Delta-phi top-k ואז סינון rho>=tau)
    return default_contrastive_selection(merged, top_k=50, tau=2.0)
contrastive = find_contrastive_features(target_prompts, control_prompts, model,
                                        catalog=catalog, select_fn=my_selection)
show_feature_candidates(contrastive)

## סריקת עוצמות (tau/mu)
STUDENT TODO: בדקו כמה ערכים ועקבו אחרי הפחתת ההתנהגות מול פגיעה כללית.

In [ ]:
import pandas as pd
from student_utils.reporting import plot_tradeoff, make_run_dir, save_score_summary
rows = []
for tau in [0.95, 0.9, 0.8]:           # STUDENT TODO
    for mu in [4.0, 8.0, 16.0]:        # STUDENT TODO
        cfg = {'tau': tau, 'mu': mu, 'linscale': True, 'use_signs': False, 'description': f'{tau}/{mu}'}
        with temporary_pisces_edit(model, gaia_feature_set, cfg):
            t = apply_scorer(df.assign(response=generate_many(tm, dataset_to_prompts(df), max_new_tokens=100)), scoring.score_sycophancy)
            g = apply_scorer(general.assign(response=generate_many(tm, dataset_to_prompts(general), max_new_tokens=100)), scoring.score_general_behavior)
        rows.append({'tau': tau, 'mu': mu,
                     'target_bad': t['target_bad_behavior'].mean(),
                     'general_coherent': g['looks_coherent'].astype(float).mean()})
sweep = pd.DataFrame(rows); sweep

In [ ]:
plot_tradeoff(sweep, 'target_bad', 'general_coherent')

In [ ]:
run_dir = make_run_dir(run_name='gaia_03_edit')
save_score_summary(run_dir, sweep)
print('saved to', run_dir)